# Book Retrieval Benchmark – Ground Truth Pipeline

Notebook này chạy toàn bộ pipeline xây dựng ground truth theo 3 bước:

| Bước | Mô tả | Output |
|------|--------|--------|
| **1. Build Indexes** | Xây TF-IDF, BM25, ChromaDB từ dataset | `models/`, `data/chroma_db/` |
| **2. Build Ground Truth** | Sinh query bằng LLM → retrieve candidates → judge relevance | `data/eval/qrels.json` |
| **3. Prune Qrels** | Xóa entries có 0 relevant book | `data/eval/qrels.json` (cleaned) |

> **Yêu cầu**: Đã có `data/processed/books.csv` và `OPENAI_API_KEY` trong `.env`.

## 0. Cấu hình & Setup

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path().resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print(f"Project root: {PROJECT_ROOT}")
print(f"Python executable: {sys.executable}")

Project root: C:\Users\LG\Desktop\khdl\AI
Python executable: c:\Users\LG\Desktop\khdl\AI\.venv\Scripts\python.exe


In [2]:
from src.config.settings import get_settings
from src.utils.logging_config import setup_logging

# Nạp settings từ .env
# get_settings() dùng @lru_cache – gọi .cache_clear() nếu muốn reload .env
settings = get_settings()
setup_logging(log_level="INFO", log_dir=settings.log_dir)

print("=" * 55)
print("SETTINGS SUMMARY")
print("=" * 55)
print(f"  Dataset           : {settings.dataset_path}")
print(f"  Max books         : {settings.max_books_to_process}")
print(f"  Queries/book      : {settings.queries_per_book}")
print(f"  Batch size (query): {settings.query_generation_batch_size}")
print(f"  GT pool/retriever : {settings.ground_truth_candidate_pool}")
print(f"  Judge workers     : {settings.judge_parallel_workers}")
print(f"  Relevance thresh  : {settings.relevance_threshold}")
print(f"  OpenAI model      : {settings.openai_model}")
print(f"  Rerank backend    : {'Jina API' if settings.rerank_use_api else 'local CrossEncoder'}")
print(f"  Eval output       : {settings.eval_output_path}")
print("=" * 55)

SETTINGS SUMMARY
  Dataset           : data\processed\books.csv
  Max books         : 100
  Queries/book      : 2
  Batch size (query): 10
  GT pool/retriever : 4
  Judge workers     : 5
  Relevance thresh  : 1
  OpenAI model      : gpt-4o-mini
  Rerank backend    : Jina API
  Eval output       : data\eval


In [3]:
# Kiểm tra API key
if not settings.openai_api_key:
    raise EnvironmentError(
        "OPENAI_API_KEY is not set.\n"
        "Add OPENAI_API_KEY to .env: OPENAI_API_KEY=sk-..."
    )
print(f"✓ OPENAI_API_KEY: {'*' * 20}{settings.openai_api_key[-6:]}")

if settings.rerank_use_api:
    print(f"✓ JINA_API_KEY: {'*' * 20}{settings.jina_api_key[-6:]}")

✓ OPENAI_API_KEY: ********************WY1cwA
✓ JINA_API_KEY: ********************S79jdb


---
## 1. Build Retrieval Indexes

Xây dựng 3 index từ dataset:
- **TF-IDF**: `models/tfidf.pkl` + `models/tfidf_matrix.npz`
- **BM25**: `models/bm25.pkl`
- **Dense (ChromaDB)**: `data/chroma_db/`

>Nếu đã build rồi và dataset không thay đổi, có thể **bỏ qua cell này**.

In [ ]:
REBUILD_INDEXES = True

if REBUILD_INDEXES:
    from src.pipelines.build_indexes import run as build_indexes

    print("Building TF-IDF, BM25 và Dense indexes…")
    build_indexes(settings=settings)
    print("\nTất cả indexes đã được build xong.")
else:
    print("bỏ qua bước build indexes.")

Building TF-IDF, BM25 và Dense indexes…
2026-06-10 16:55:14 | INFO     | src.pipelines.build_indexes | ============================================================
2026-06-10 16:55:14 | INFO     | src.pipelines.build_indexes | BUILD INDEXES PIPELINE
2026-06-10 16:55:14 | INFO     | src.pipelines.build_indexes | ============================================================
2026-06-10 16:55:14 | INFO     | src.pipelines.build_indexes | Loading dataset from 'data\processed\books.csv'…
2026-06-10 16:55:15 | INFO     | src.pipelines.build_indexes | Capping corpus: 12688 → 100 books (MAX_BOOKS_TO_PROCESS=100)
2026-06-10 16:55:15 | INFO     | src.pipelines.build_indexes | Cleaning description text for 100 books…
2026-06-10 16:55:15 | INFO     | src.pipelines.build_indexes | Final corpus size: 100 books
2026-06-10 16:55:15 | INFO     | src.pipelines.build_indexes | Building TF-IDF index…
2026-06-10 16:55:15 | INFO     | src.retrieval.tfidf_retriever | Building TF-IDF index: 100 documents, max_f

c:\Users\LG\Desktop\khdl\AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-06-10 16:55:37 | INFO     | src.retrieval.dense_retriever | Building dense index: 100 documents with model 'BAAI/bge-small-en-v1.5'
2026-06-10 16:55:37 | INFO     | src.retrieval.dense_retriever | Model cache dir: C:\Users\LG\Desktop\khdl\AI\models\bge_cache
2026-06-10 16:55:37 | INFO     | sentence_transformers.base.model | No device provided, using cpu
2026-06-10 16:55:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/modules.json "HTTP/1.1 307 Temporary Redirect"


2026-06-10 16:55:39 | WARNING  | huggingface_hub.utils._http | Warning: You are sending unauthenticated requests to the HF Hub. Please set a HF_TOKEN to enable higher rate limits and faster downloads.
2026-06-10 16:55:39 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/modules.json "HTTP/1.1 200 OK"
2026-06-10 16:55:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/config_sentence_transformers.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 16:55:40 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b44b94cc5a9bb96e735b38267a/config_sentence_transformers.json "HTTP/1.1 200 OK"
2026-06-10 16:55:40 | INFO     | sentence_transformers.base.model | Loading SentenceTransformer model from BAAI/bge-small-en-v1.5.
2026-06-10 16:55:40 | INFO     | httpx | HTTP Request:

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 1244.01it/s]


2026-06-10 16:55:44 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 16:55:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 16:55:45 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/video_preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 16:55:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 16:55:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/tokenizer_config.json "HTTP/1.1 307 Temporary Redirect"
2026-06-10 16:55:46 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/BAAI/bge-small-en-v1.5/5c38ec7c405ec4b

Batches: 100%|██████████| 2/2 [00:10<00:00,  5.25s/it]


2026-06-10 16:56:03 | INFO     | src.retrieval.dense_retriever | Deleted existing collection 'books'


Upserting to ChromaDB: 100%|██████████| 1/1 [00:00<00:00,  3.47it/s]

2026-06-10 16:56:03 | INFO     | src.retrieval.dense_retriever | Dense index built: 100 documents in collection 'books'
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes | Dense index complete ✓
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes | ============================================================
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes | ALL INDEXES BUILT SUCCESSFULLY
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes |   TF-IDF model: models\tfidf.pkl
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes |   TF-IDF matrix: models\tfidf_matrix.npz
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes |   BM25 index: models\bm25.pkl
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes |   ChromaDB: data\chroma_db (collection: books)
2026-06-10 16:56:03 | INFO     | src.pipelines.build_indexes | ============================================================

Tất cả indexes đã được build xong.


In [6]:
index_files = {
    "TF-IDF model": Path(settings.tfidf_model_path),
    "TF-IDF matrix": Path(settings.tfidf_matrix_path),
    "BM25 index": Path(settings.bm25_index_path),
    "ChromaDB": Path(settings.chroma_path),
}
all_ok = True
for name, path in index_files.items():
    exists = path.exists()
    status = "" if exists else "MISSING"
    print(f"  {status}  {name}: {path}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("Một số index files bị thiếu.")

    TF-IDF model: models\tfidf.pkl
    TF-IDF matrix: models\tfidf_matrix.npz
    BM25 index: models\bm25.pkl
    ChromaDB: data\chroma_db


---
## 2. Build Ground Truth (qrels.json)

Pipeline:
```
Books
  ↓
LLM sinh queries (QUERIES_PER_BOOK mỗi sách)
  ↓
[Concurrent] 5 retrievers × GROUND_TRUTH_CANDIDATE_POOL candidates
  TF-IDF | BM25 | Dense | Hybrid RRF | Reranking
  ↓  union (dedup by ISBN)
LLM judge relevance (0/1/2) ← JUDGE_PARALLEL_WORKERS threads
  ↓
qrels.json  ← lưu incremental sau mỗi query (Ctrl+C safe)
```

> **Checkpoint/Resume**: Lưu lại kết quả khi cell bị gián đoạn
>
> **Append mode**: Đặt `APPEND_MODE=True` để gộp kết quả vào file cũ (không skip query_id đã done).

In [10]:
# Override settings
POOL_SIZE_PER_RETRIEVER        = 10
BATCH_SIZE                     = None
QUERIES_PER_BOOK               = None
APPEND_MODE                    = True

# Áp dụng override (chỉ khi khác None)
if POOL_SIZE_PER_RETRIEVER is not None:
    settings.ground_truth_candidate_pool = POOL_SIZE_PER_RETRIEVER
    print(f"[override] ground_truth_candidate_pool = {POOL_SIZE_PER_RETRIEVER}")
if BATCH_SIZE is not None:
    settings.query_generation_batch_size = BATCH_SIZE
    print(f"[override] query_generation_batch_size = {BATCH_SIZE}")
if QUERIES_PER_BOOK is not None:
    settings.queries_per_book = QUERIES_PER_BOOK
    print(f"[override] queries_per_book = {QUERIES_PER_BOOK}")

print()
print("Effective settings:")
print(f"  pool_size_per_retriever        = {settings.ground_truth_candidate_pool}")
print(f"  batch_size                     = {settings.query_generation_batch_size}")
print(f"  queries_per_book               = {settings.queries_per_book}")
print(f"  append_mode                    = {APPEND_MODE}")
print(f"  judge_workers                  = {settings.judge_parallel_workers}")

[override] ground_truth_candidate_pool = 10

Effective settings:
  pool_size_per_retriever        = 10
  batch_size                     = 10
  queries_per_book               = 2
  append_mode                    = True
  judge_workers                  = 5


In [11]:
from src.pipelines.build_ground_truth import run as build_ground_truth

print("Starting ground truth pipeline…")
print("(Ctrl+C để dừng – tiến độ được lưu sau mỗi query)\n")

qrels = build_ground_truth(settings=settings, append=APPEND_MODE)

print(f"\nGround truth hoàn tất: {len(qrels)} qrel entries.")
print(f"  Queries có ≥1 relevant: {sum(1 for q in qrels if q.relevant_isbns)}")
print(f"  Queries có 0 relevant : {sum(1 for q in qrels if not q.relevant_isbns)}")

Starting ground truth pipeline…
(Ctrl+C để dừng – tiến độ được lưu sau mỗi query)

2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | ============================================================
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | BUILD GROUND TRUTH PIPELINE
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | mode: append
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | ============================================================
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | Checkpoint loaded: 205 qrels already done. Resuming from there.
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | Append mode: keeping 205 existing qrels, done-filter cleared – all new queries will be processed.
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth | Loading dataset from 'data\processed\books.csv'…
2026-06-10 17:02:32 | INFO     | src.pipelines.build_ground_truth |

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 5073.96it/s]


2026-06-10 17:04:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 17:04:49 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 17:04:50 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/processor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 17:04:50 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 17:04:50 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Found"
2026-06-10 17:04:50 | INFO     | httpx | HTTP Request: HEAD https://huggingface.co/BAAI/bge-small-en-v1.5/resolve/main/preprocessor_config.json "HTTP/1.1 404 Not Fo

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

2026-06-10 17:04:53 | INFO     | httpx | HTTP Request: GET https://huggingface.co/api/models/BAAI/bge-small-en-v1.5 "HTTP/1.1 200 OK"


...

2026-06-10 18:25:10 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:11 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 0: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:11 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:11 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 3: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:11 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:11 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring fa

Batches: 100%|██████████| 1/1 [00:00<00:00,  5.17it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  4.34it/s]


2026-06-10 18:25:26 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:26 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 0: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:26 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 200 OK"
2026-06-10 18:25:26 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:26 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 2: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:27 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.45it/s]

Batches: 100%|██████████| 1/1 [00:00<00:00,  3.52it/s]


2026-06-10 18:25:41 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:41 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 4: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:41 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:41 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring failed for candidate 3: Client error '429 Too Many Requests' for url 'https://api.jina.ai/v1/rerank'
For more information check: https://developer.mozilla.org/en-US/docs/Web/HTTP/Status/429
2026-06-10 18:25:41 | INFO     | httpx | HTTP Request: POST https://api.jina.ai/v1/rerank "HTTP/1.1 429 Too Many Requests"
2026-06-10 18:25:41 | ERROR    | src.retrieval.rerank_retriever | Jina API scoring fa

In [12]:
# Xem phân bố số relevant books mỗi query
from collections import Counter

dist = Counter(len(q.relevant_isbns) for q in qrels)
print("Phân bố số relevant books / query:")
for cnt in sorted(dist):
    bar = "█" * min(dist[cnt], 40)
    print(f"  {cnt:>3} relevant: {bar} ({dist[cnt]} queries)")

Phân bố số relevant books / query:
    0 relevant: █████████ (9 queries)
    1 relevant: ████████████████████████████████████████ (64 queries)
    2 relevant: ████████████████████████████████████████ (58 queries)
    3 relevant: █████████████████████████████████████ (37 queries)
    4 relevant: █████████████████████████████████████ (37 queries)
    5 relevant: ██████████████████████████ (26 queries)
    6 relevant: █████████████████████ (21 queries)
    7 relevant: ███████████████████████████ (27 queries)
    8 relevant: █████████████████ (17 queries)
    9 relevant: ███████████████ (15 queries)
   10 relevant: ███████████████████ (19 queries)
   11 relevant: ████ (4 queries)
   12 relevant: █████████ (9 queries)
   13 relevant: █ (1 queries)
   14 relevant: █████ (5 queries)
   15 relevant: ██████ (6 queries)
   16 relevant: ████ (4 queries)
   17 relevant: ████ (4 queries)
   18 relevant: █ (1 queries)
   19 relevant: ██ (2 queries)
   20 relevant: ████ (4 queries)
   21 relevant: █ 

---
## 3. Prune Qrels

Xóa các entries có `relevant_isbns = []` — những entries này không đóng góp gì vào evaluation metrics và làm tăng ảo số query.

Quy trình:
1. **Dry run** (mặc định): chỉ xem thống kê, không sửa file
2. **Execute**: tạo backup `.bak.json` → ghi đè `qrels.json`

In [13]:
import json
import shutil

qrels_path = Path(settings.eval_output_path) / "qrels.json"

with open(qrels_path, encoding="utf-8") as fh:
    raw = json.load(fh)

total  = len(raw)
kept   = [e for e in raw if e.get("relevant_isbns")]
pruned = total - len(kept)

print(f"qrels.json: {total} entries")
print(f"Với relevant books  : {len(kept)}")
print(f"Không có relevant   : {pruned}")

qrels.json: 382 entries
Với relevant books  : 373
Không có relevant   : 9


In [14]:
EXECUTE_PRUNE = False   # True: thực sự xóa
CREATE_BACKUP = True    # False: bỏ qua tạo file backup

if pruned == 0:
    print("Không có entries nào cần xóa.")
elif not EXECUTE_PRUNE:
    print(f"[dry-run] Sẽ xóa {pruned} entries có 0 relevant books.")
else:
    if CREATE_BACKUP:
        backup_path = qrels_path.with_suffix(".bak.json")
        shutil.copy2(qrels_path, backup_path)
        print(f"  Backup: {backup_path}")

    tmp_path = qrels_path.with_suffix(".tmp")
    with open(tmp_path, "w", encoding="utf-8") as fh:
        json.dump(kept, fh, indent=2, default=str)
    tmp_path.replace(qrels_path)

    print(f"  Đã xóa {pruned} entries.")
    print(f"  qrels.json hiện có: {len(kept)} entries.")

[dry-run] Sẽ xóa 9 entries có 0 relevant books.


---
## 4. Kiểm tra kết quả cuối

In [15]:
with open(qrels_path, encoding="utf-8") as fh:
    final = json.load(fh)

print(f"{'='*50}")
print("QRELS SUMMARY")
print(f"{'='*50}")
print(f"  Total entries       : {len(final)}")
print(f"  With ≥1 relevant    : {sum(1 for e in final if e['relevant_isbns'])}")
print(f"  Avg relevant/query  : {sum(len(e['relevant_isbns']) for e in final) / max(len(final), 1):.2f}")
print(f"  Output file         : {qrels_path}")
print(f"{'='*50}")

# Xem 3 ví dụ
print("\nVí dụ 3 qrel entries đầu:")
for entry in final[:3]:
    print(f"  query_id     : {entry['query_id']}")
    print(f"  query        : {entry['query'][:80]}")
    print(f"  source_isbn  : {entry['source_isbn']}")
    print(f"  relevant ({len(entry['relevant_isbns'])}): {entry['relevant_isbns'][:5]}")
    print()

QRELS SUMMARY
  Total entries       : 382
  With ≥1 relevant    : 373
  Avg relevant/query  : 5.92
  Output file         : data\eval\qrels.json

Ví dụ 3 qrel entries đầu:
  query_id     : q_9780002188319_0
  query        : World Cup statistics
  source_isbn  : 9780002188319
  relevant (1): ['9780002188319']

  query_id     : q_9780002188319_1
  query        : history of soccer tournaments
  source_isbn  : 9780002188319
  relevant (1): ['9780002188319']

  query_id     : q_9780002188319_2
  query        : detailed match reports and player statistics from the World Cup finals
  source_isbn  : 9780002188319
  relevant (1): ['9780002188319']

